In [ ]:
#IMPORTING DATA
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

raw_pilot_data = pd.read_csv(r"C:\Users\haran\Downloads\inlab_pilot_109_source.csv")


In [ ]:
#DATA CLEANING
#flagging practice trials
raw_pilot_data['trial_index'] = pd.to_numeric(raw_pilot_data['trial_index'], errors='coerce')

raw_pilot_data['block_marker'] = (
        raw_pilot_data['stimulus']
        .str.contains(r"Blokk\s*1\s*kezdődik", case=False, regex=True, na=False)
    )

raw_pilot_data['start_seen'] = (
    raw_pilot_data
    .groupby('subj_code')['trial_index']
    .transform(lambda x: x.eq(0).astype(int).cummax())
)

raw_pilot_data['marker_seen'] = (
    raw_pilot_data.groupby('subj_code')['block_marker']
    .transform(lambda x: x.astype(int).cummax())
)

raw_pilot_data['trial_exp'] = np.where(
    (raw_pilot_data['start_seen'] == 1) & (raw_pilot_data['marker_seen'] == 0),
    'practice',
    'experimental'
)

In [ ]:
#flagging previous trials
raw_pilot_data['prev_congruency'] = raw_pilot_data['congruency'].shift(4)
raw_pilot_data['prev_correct'] = raw_pilot_data['correct'].shift(4)

raw_pilot_data['correct'] = raw_pilot_data['correct'].map({'true': 1, 'false': 0})
raw_pilot_data['prev_correct'] = raw_pilot_data['prev_correct'].map({'true': 1, 'false': 0})

In [ ]:
#flagging first trials
raw_pilot_data['first_trial'] = raw_pilot_data['stimulus'].shift(4).str.contains(r"Blokk \d+ kezdődik", regex = True)
raw_pilot_data['first_trial'] = raw_pilot_data['first_trial'].astype(int)

raw_pilot_data['rt'] = pd.to_numeric(raw_pilot_data['rt'], errors='coerce')

In [ ]:
#filtering subjects with accuracy below <60%
accuracy = (
    raw_pilot_data[(raw_pilot_data['task'] == 'probe') &
                   (raw_pilot_data['trial_exp'] != 'practice') &
                   (raw_pilot_data['first_trial'] != 1)
                   ]
    .groupby('subj_code')['correct']
    .mean()
    .reset_index(name='all_accuracy')
)

raw_pilot_data = (
    raw_pilot_data
    .merge(accuracy, on='subj_code', how='left')
)

raw_pilot_data = raw_pilot_data[
    raw_pilot_data['all_accuracy'] > 0.60
]

In [ ]:
#filtering data
filtered_pilot_data = (
    raw_pilot_data[(raw_pilot_data['task'] == 'probe') &
                   (raw_pilot_data['correct'] == 1) &
                   (raw_pilot_data['prev_correct'] == 1) &
                   (raw_pilot_data['trial_exp'] != 'practice') &
                   (raw_pilot_data['first_trial'] != 1) &
                   (raw_pilot_data['rt'] >= 150)
    ]
)

processed_pilot_data = filtered_pilot_data[
    ['rt', 'subj_code', 'task', 'congruency', 'color', 'monetary', 'experiment', 'prev_congruency', 'all_accuracy']
]

In [ ]:
#DESCRIPTIVE STATISTICS on RT values
def descriptives(x, na_omit=False):
    x = np.array(x, dtype=int)
    
    if na_omit:
        x = x[~np.isnan(x)]
    
    minimum = np.min(x)
    maximum = np.max(x)
    mean = np.mean(x)
    median = np.median(x)
    n = len(x)
    stds = np.std(x, ddof=1)
    
    skew = 3 * (mean - median) / stds
    kurt = (n * np.sum((x - mean)**4)) / (np.sum((x - mean)**2)**2) - 3
    
    return {
        'min': minimum,
        'max': maximum,
        'mean': mean,
        'median': median,
        'sd': stds,
        'skewness': skew,
        'kurtosis': kurt
    }

mean_rt = (
    processed_pilot_data
    .pivot_table(
        index='subj_code',
        columns='congruency',
        values='rt',
        aggfunc='mean'
    )
    .rename(columns={
        'congruent': 'mean_RT_congruent',
        'incongruent': 'mean_RT_incongruent'
    })
    .reset_index()
)

rt_cols = ['mean_RT_congruent', 'mean_RT_incongruent']
rt_descriptives = mean_rt[rt_cols].apply(
    lambda col: pd.Series(descriptives(col, na_omit=True))
).T

print(rt_descriptives)

In [ ]:
#SUBJECT CSE VALUES
subject_cse = (
    processed_pilot_data
    .groupby(['subj_code', 'congruency', 'prev_congruency'], as_index=False)
    .agg(mean_RT=('rt', 'mean'))
    .pivot(columns = ['prev_congruency', 'congruency'], index = 'subj_code', values = 'mean_RT')
)

subject_cse.columns = [f'{prev}-{curr}' for prev, curr in subject_cse.columns]

subject_cse['CSE'] = (
    (subject_cse['congruent-incongruent'] - subject_cse['congruent-congruent']) -
        (subject_cse['incongruent-incongruent'] - subject_cse['incongruent-congruent'])
)

In [ ]:
#ADDING MONETARY VALUES
processed_pilot_data['prev_color'] = processed_pilot_data.groupby('subj_code')['color'].shift(1)

def prev_color_to_monetary(x):
    if x == 'red':
        return 'loss'
    elif x == 'green':
        return 'gain'
    elif x in ['magenta', 'blue', 'yellow']:
        return 'neutral'
    else:
        return None

processed_pilot_data['prev_color'] = processed_pilot_data['prev_color'].apply(prev_color_to_monetary)

In [ ]:
#PARTICIPANT CSE (MONETARY)
cse_color_df = (
    processed_pilot_data
    .groupby(['subj_code', 'congruency', 'prev_congruency', 'prev_color'], as_index=False)
    .agg(
        N=('rt', 'count'),
        mean_rt=('rt', 'mean')
    )
)

wide_color_df = cse_color_df.pivot(
    index='subj_code',
    columns=['prev_color', 'prev_congruency', 'congruency'],
    values='mean_rt'
).reset_index()

wide_color_df.columns = [
    '_'.join(col).strip() if isinstance(col, tuple) else col
    for col in wide_color_df.columns.values
]

subject_color_cse = wide_color_df.copy()

subject_color_cse['CSE_loss'] = (
    (subject_color_cse['loss_congruent_incongruent'] - subject_color_cse['loss_congruent_congruent']) -
    (subject_color_cse['loss_incongruent_incongruent'] - subject_color_cse['loss_incongruent_congruent'])
)

subject_color_cse['CSE_gain'] = (
    (subject_color_cse['gain_congruent_incongruent'] - subject_color_cse['gain_congruent_congruent']) -
    (subject_color_cse['gain_incongruent_incongruent'] - subject_color_cse['gain_incongruent_congruent'])
)

subject_color_cse['CSE_neutral'] = (
    (subject_color_cse['neutral_congruent_incongruent'] - subject_color_cse['neutral_congruent_congruent']) -
    (subject_color_cse['neutral_incongruent_incongruent'] - subject_color_cse['neutral_incongruent_congruent'])
)

In [22]:
#HYPOTHESIS TESTING (FREQUENTIST)
#H1: Mean RT will be higher in incongruent trials than congruent trials, indicating the congruency effect (CE).


#assumption check
assumption_h1 = mean_rt['mean_RT_incongruent'] - mean_rt['mean_RT_congruent']
h1normal, h1normalp = stats.shapiro(assumption_h1)

print(f"W = {h1normal}")
print(f"p = {h1normalp}")

#paired samples t-test
t_stat, p_value = stats.ttest_rel(
    mean_rt['mean_RT_congruent'],
    mean_rt['mean_RT_incongruent'],
    alternative='less'
)

df = len(mean_rt) - 1

print(f"t({df}) = {t_stat:.3f}, p = {p_value:.3f}")

W = 0.9419876558159596
p = 0.6664832395649716
t(3) = -4.308, p = 0.012


In [21]:
#assumption checks for CSE

assumption_loss, assumption_loss_p = stats.shapiro(subject_color_cse["CSE_loss"])
assumption_gain, assumption_gain_p = stats.shapiro(subject_color_cse["CSE_gain"])
assumption_neutral, assumption_neutral_p = stats.shapiro(subject_color_cse["CSE_neutral"])

'''nt("Loss:", (W = {assumption_loss}, (p = assumption_loss_p)
print("Gain:", assumption_gain, assumption_gain_p)
print("Neutral:", assumption_neutral, assumption_neutral_p)'''

print("Loss:")
print(f"W = {assumption_loss}")
print(f"p = {assumption_loss_p}")
print("Gain:")
print(f"W = {assumption_gain}")
print(f"p = {assumption_gain_p}")
print("Neutral:")
print(f"W = {assumption_neutral}")
print(f"p = {assumption_neutral_p}")


Loss:
W = 0.7854067144417117
p = 0.0785213436243739
Gain:
W = 0.972681120303516
p = 0.8580095717400709
Neutral:
W = 0.7169758004871369
p = 0.017990180754257467


In [23]:
#H2: Reaction times will be modulated by the congruence of the current and previous trials (CSE).
#one sample t-test

t_stat2, p_value2 = stats.ttest_1samp(
    a = subject_cse['CSE'], popmean = 0, alternative = 'greater')

df2 = len(subject_cse['CSE']) - 1

print(f"t({df2}) = {t_stat2:.3f}, p = {p_value2:.3f}")

t(3) = 1.348, p = 0.135


In [24]:
#H2.1: CSE for gain condition
t_stat21, p_value21 = stats.ttest_1samp(
    a = subject_color_cse['CSE_gain'], popmean = 0, alternative = 'greater')

df21 = len(subject_color_cse['CSE_gain']) - 1

print(f"t({df21}) = {t_stat21:.3f}, p = {p_value21:.3f}")


t(3) = 0.540, p = 0.313


In [ ]:
#H2.2: CSE for loss condition
t_stat22, p_value22 = stats.ttest_1samp(
    a = subject_color_cse['CSE_loss'], popmean = 0, alternative = 'greater')

df22 = len(subject_color_cse['CSE_loss']) - 1

print(f"t({df22}) = {t_stat22:.3f}, p = {p_value22:.3f}")


In [ ]:
#H2.3: CSE for neutral condition
t_stat23, p_value23 = stats.ttest_1samp(
    a = subject_color_cse['CSE_neutral'], popmean = 0, alternative = 'greater')

df23 = len(subject_color_cse['CSE_neutral']) - 1

print(f"t({df23}) = {t_stat23:.3f}, p = {p_value23:.3f}")

In [27]:
#H3: CSE will be stronger in the loss condition compared to the neutral condition
#paired samples t-test

#assumption check
h3assumption = subject_color_cse['CSE_loss'] - subject_color_cse['CSE_neutral']
h3normal, h3normalp = stats.shapiro(h3assumption)

print(f"W = {h3normal}")
print(f"p = {h3normalp}")

t_stat3, p_value3 = stats.ttest_rel(
    subject_color_cse['CSE_loss'],
    subject_color_cse['CSE_neutral'],
    alternative = 'greater'
)

df3 = len(subject_color_cse) - 1

print(f"t({df3}) = {t_stat3:.3f}, p = {p_value3:.3f}")

W = 0.9101626382939104
p = 0.48326422944901926
t(3) = 0.182, p = 0.433


In [29]:
#H4: CSE will be weaker in the gain condition compared to the neutral condition

#assumption check

h4assumption = subject_color_cse['CSE_neutral'] - subject_color_cse['CSE_gain']
h4normal, h4normalp = stats.shapiro(h4assumption)

print(f"W = {h4normal}")
print(f"p = {h4normalp}")

t_stat4, p_value4 = stats.ttest_rel(
    subject_color_cse['CSE_gain'],
    subject_color_cse['CSE_neutral'],
    alternative = 'less'
)

df4 = len(subject_color_cse) - 1

print(f"t({df4}) = {t_stat4:.3f}, p = {p_value4:.3f}")

W = 0.9435273937338539
p = 0.6759477976694257
t(3) = -0.439, p = 0.345
